# Sentiment scores by aspect for one game

Set `APP_ID` and run all cells. Load cached reviews and deduplicate recommendation IDs. Use the analyzer's base and learned sentiment lists, including the current `LEARNT_EXTENDED` negative list.

Each positive sentiment occurrence contributes +1 and each negative occurrence −1 to **every aspect mentioned in that sentence**. Nearby negation flips that contribution. Repeated aspect keywords do not multiply contributions. This follows sentence-level co-occurrence: a sentence discussing several aspects assigns its sentiment to all of them, rather than resolving which adjective describes which aspect.

Negation uses `ReviewParser.is_negated`, looking back three tokens before each sentiment word. Preserve stop words and punctuation for that check; punctuation, contrast words, and “and” stop its scope. Examples: `not good` → −1; `isn't bad` → +1. This heuristic does not interpret sarcasm or complex grammatical scope.

Print positive and negative contribution counts and their net semantic score for every aspect. A zero score may mean no evidence or balanced evidence; the counts distinguish them. Whole-review Steam labels are not used to score sentences.


In [5]:
import re
import sys
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src").is_dir() and (path / "notebooks").is_dir()), None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start from the project root or notebooks folder")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.csv_io import load_reviews
from src.review_parser import ReviewParser
from src.semantic_analyzer import SemanticAnalyzer

APP_ID = 2567870
NEGATION_WINDOW = 3
parser = ReviewParser()
positive_words = set(SemanticAnalyzer.POSITIVE_WORDS)
negative_words = set(SemanticAnalyzer.NEGATIVE_WORDS)
for name in ("LEARNT_POSITIVE_WORDS", "EXTENDED_POSITIVE_WORDS"):
    positive_words.update(getattr(SemanticAnalyzer, name, []))
for name in ("LEARNT_NEGATIVE_WORDS", "EXTENDED_NEGATIVE_WORDS", "LEARNT_EXTENDED"):
    negative_words.update(getattr(SemanticAnalyzer, name, []))
analyzer = SemanticAnalyzer(positive_words=positive_words, negative_words=negative_words)
if analyzer.positive_words & analyzer.negative_words:
    raise ValueError("A semantic word appears in both polarity lists")


In [6]:
def score_aspects(reviews):
    scores = {aspect: {"positive": 0, "negative": 0, "semantic_score": 0,
                       "matched_sentences": 0} for aspect in analyzer.aspects}
    evidence = []
    unique_reviews = {review.recommendationid: review for review in reviews}
    for review in unique_reviews.values():
        for sentence in parser.split_sentences(review.review):
            # Retain punctuation as scope boundaries; retain contractions as words.
            tokens = re.findall(r"\b\w+(?:'\w+)*\b|[,;:.!?]",
                                sentence.replace("’", "'").lower())
            aspects = [aspect for aspect, words in analyzer.aspects.items()
                       if set(tokens).intersection(words)]
            if not aspects:
                continue
            matches = []
            for index, token in enumerate(tokens):
                if token in analyzer.positive_words:
                    polarity = 1
                elif token in analyzer.negative_words:
                    polarity = -1
                else:
                    continue
                negated = parser.is_negated(tokens, index, NEGATION_WINDOW)
                contribution = -polarity if negated else polarity
                matches.append({"word": token, "negated": negated, "score": contribution})
                for aspect in aspects:
                    scores[aspect]["positive" if contribution > 0 else "negative"] += 1
                    scores[aspect]["semantic_score"] += contribution
            if matches:
                for aspect in aspects:
                    scores[aspect]["matched_sentences"] += 1
                evidence.append({"recommendationid": review.recommendationid,
                                 "sentence": sentence, "aspects": aspects, "matches": matches})
    return scores, evidence


In [7]:
reviews = load_reviews(APP_ID)
if not reviews:
    raise ValueError(f"No saved reviews found for app {APP_ID}; fetch them first")
aspect_scores, evidence = score_aspects(reviews)
print(f"App {APP_ID}: analyzed {len({review.recommendationid for review in reviews})} unique reviews")
print(f"{'Aspect':16} {'Positive':>10} {'Negative':>10} {'Semantic score':>16} {'Sentences':>10}")
for aspect, counts in aspect_scores.items():
    print(f"{aspect:16} {counts['positive']:10} {counts['negative']:10} "
          f"{counts['semantic_score']:+16} {counts['matched_sentences']:10}")


App 2567870: analyzed 100 unique reviews
Aspect             Positive   Negative   Semantic score  Sentences
gameplay                  4          1               +3          4
story                     0          1               -1          1
graphics                  0          0               +0          0
audio                     0          0               +0          0
performance               3          0               +3          1
world                     0          0               +0          0
value                     3          0               +3          2
multiplayer              12          0              +12          9


In [8]:
# Inspect which words and negations contributed to the scores.
evidence[:20]


[{'recommendationid': '172418432',
  'sentence': 'You are looking at a cooperative version of a series of games similar to a "Difficult Game About Climbing" which includes adjustable difficulty and simple checkpoints for those who don\'t want to suffer, as well as increased difficulty for those who enjoy a challenge.',
  'aspects': ['gameplay', 'multiplayer'],
  'matches': [{'word': 'enjoy', 'negated': False, 'score': 1}]},
 {'recommendationid': '172418432',
  'sentence': "It's just okay - in a group, you'll probably have fun, but like with any cooperative game, the game itself entertains you very little.",
  'aspects': ['multiplayer'],
  'matches': [{'word': 'fun', 'negated': False, 'score': 1}]},
 {'recommendationid': '196927472',
  'sentence': '☐ Bad story',
  'aspects': ['story'],
  'matches': [{'word': 'bad', 'negated': False, 'score': -1}]},
 {'recommendationid': '187237744',
  'sentence': 'The physics-based gameplay can be hilariously frustrating, especially as you and your frie